# 93.75% overlap parameter sweep

Compare rPIE, LSQML, GM-rPIE, and GM-MAGPIE at η=0.1, batch 27, and 200 epochs. Each PIE method searches nine alpha pairs; LSQML searches sixteen damping–σ pairs. Selection uses the lowest final object NMSE.

These cells inspect the included publication records on CPU. The optional rerun is disabled.

In [ ]:
STUDY = "overlap"
OVERLAP = 0.9375
RUN_RECONSTRUCTIONS = False


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from magpie_publication import load_index, select_cases, read_history, run_study

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "results" / "index.json").is_file() and (p / "pyproject.toml").is_file())
index = load_index(root=ROOT)
endpoint_csv = ROOT / index["studies"][STUDY]["endpoint_csv"]
endpoint_table = pd.read_csv(endpoint_csv)


In [ ]:
METHODS = ["rpie", "lsqml", "gm_rpie", "gm_magpie"]
LABELS = dict(zip(METHODS, ["rPIE", "LSQML", "GM-rPIE", "GM-MAGPIE"]))
COLORS = dict(zip(METHODS, ["#4477AA", "#EE7733", "#228833", "#AA3377"]))
PARAMETER_LABELS = {"object_alpha": "α_O", "probe_alpha": "α_P", "damping": "damping", "sigma": "σ", "beta": "β"}

def case_table(cases):
    rows = []
    for case in sorted(cases, key=lambda c: (c.get("batch_size") or 0, c.get("eta") or 0, METHODS.index(c["method"]))):
        parameters = ", ".join(f"{PARAMETER_LABELS.get(k, k)}={v:g}" for k, v in case["parameters"].items())
        rows.append({"Method": LABELS[case["method"]], "Parameters": parameters,
                     "η": case.get("eta"), "Batch": case["batch_size"],
                     "Epochs": case["epochs_completed"], "Status": case["status"],
                     "Object NMSE": case.get("object_nmse")})
    return pd.DataFrame(rows).dropna(axis=1, how="all")


## Selected parameter pairs

Object error is the normalized squared difference from the truth after complex-scale and affine-phase alignment on the common scanned region.

In [ ]:
cases = select_cases(STUDY, overlap=OVERLAP, eta=0.1, batch_size=27, selected=True, root=ROOT)
assert len(cases) == 4
display(case_table(cases))


In [ ]:
def plot_object_histories(cases, *, group_by=None, values=None):
    values = [None] if group_by is None else values
    fig, axes = plt.subplots(1, len(values), figsize=(5.8 if len(values) == 1 else 10, 3.2),
                             sharey=True, squeeze=False)
    for axis, value in zip(axes[0], values):
        subset = cases if group_by is None else [c for c in cases if c[group_by] == value]
        for case in sorted(subset, key=lambda c: METHODS.index(c["method"])):
            history = read_history(case, root=ROOT)
            points = history[["epoch", "object_nmse"]].apply(pd.to_numeric, errors="coerce").dropna()
            points = points[points["object_nmse"] > 0]
            axis.semilogy(points["epoch"], points["object_nmse"],
                          label=LABELS[case["method"]], color=COLORS[case["method"]])
        if group_by == "eta":
            axis.set_title(f"η = {value:g}")
        elif group_by == "batch_size":
            axis.set_title(f"Batch {value}")
        axis.set_xlabel("Epoch")
        axis.grid(True, which="both", alpha=0.2)
    axes[0, 0].set_ylabel("Object error (NMSE)")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
    fig.tight_layout(rect=(0, 0.12, 1, 1))
    plt.show()


In [ ]:
plot_object_histories(cases)


## Paper figures

In [ ]:
display(Image(filename=str(ROOT / "results" / "figures" / "numerical_overlap_convergence.png")))


## All parameter-pair endpoints

The complete grid is retained in the endpoint CSV, including numerical outcomes that were not selected.

In [ ]:
grid_cases = select_cases(STUDY, overlap=OVERLAP, eta=0.1, batch_size=27, root=ROOT)
display(case_table(grid_cases))


## Optional reconstruction rerun

Set `RUN_RECONSTRUCTIONS = True` to repeat this study on a CUDA GPU. Rerun outputs are separate from the archived publication records displayed above.

In [ ]:
if RUN_RECONSTRUCTIONS:
    rerun = run_study("overlap", overlap=OVERLAP, root=ROOT)
    print("Rerun exit code:", rerun.returncode)
else:
    print("Rerun disabled; the included publication results are shown above.")
